In [3]:
import pickle
import numpy as np

In [4]:
ohe_sex = pickle.load(open('models/ohe_sex.pkl', 'rb'))
ohe_embarked = pickle.load(open('models/ohe_embarked.pkl', 'rb'))
clf = pickle.load(open('models/clf.pkl', 'rb'))

In [5]:
# Assume user input
# Pclass/gender/age/SibSp/Parch/Fare/Embarked
test_input = np.array([2,'male',31.0,0,0,10.5,'S'], dtype=object).reshape((1,7))

In [6]:
test_input

array([[2, 'male', 31.0, 0, 0, 10.5, 'S']], dtype=object)

In [7]:
test_input_sex = ohe_sex.transform(test_input[:,1].reshape(1,1))

P:\Anaconda\envs\ml-env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [8]:
test_input_embarked = ohe_embarked.transform(test_input[:,-1].reshape(1,1))

P:\Anaconda\envs\ml-env\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [9]:
test_input_age = test_input[:,2].reshape(1,1)

In [10]:
test_input_transformed = np.concatenate((test_input[:,[0,3,4,5]], test_input_age,test_input_sex,test_input_embarked), axis=1)

In [11]:
test_input_transformed.shape

(1, 11)

In [12]:
clf.predict(test_input_transformed)

array([1])

# With ML Pipeline

In [13]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import  SimpleImputer
from sklearn.compose import  ColumnTransformer
from  sklearn.metrics import accuracy_score


In [14]:
df = pd.read_csv('train.csv')

In [15]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [16]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'], inplace=True)

In [17]:
df.sample(5)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
2,1,3,female,26.0,0,0,7.925,S
107,1,3,male,NaN,0,0,7.775,S
110,0,1,male,47.0,0,0,52.000,S
543,1,2,male,32.0,1,0,26.000,S
0,0,3,male,22.0,1,0,7.250,S


In [18]:
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=42)

In [19]:
X_train.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [20]:
trf1 = ColumnTransformer(
    transformers=[
        ('impute_age', SimpleImputer(), [2]),
        ('impute_embarked', SimpleImputer(strategy='most_frequent'), [6]),
    ],
    remainder='passthrough'
)

In [21]:
trf2 = ColumnTransformer(
    transformers=[
        ('ohe_sex_embarked', OneHotEncoder(handle_unknown='ignore',sparse_output=False), [1,6]),
    ],
    remainder='passthrough'
)

In [22]:
trf3 = ColumnTransformer(
    transformers=[
        ('scale',MinMaxScaler(),slice(0,10))
    ]
)

In [23]:
# feature selection
trf4  = SelectKBest(score_func=chi2, k=8)

In [24]:
trf5 = DecisionTreeClassifier()

In [25]:
pipe = Pipeline(
    [
        ('trf1', trf1),
        ('trf2', trf2),
        ('trf3', trf3),
        ('trf4', trf4),
        ('trf5', trf5)
    ]
)

## Pipeline VS Make_pipeline

Pipeline requires naming of steps, make_pipeline doesnt

(Same applies to Column-transformer vs make_column_transformer )

In [26]:
# Alternate syntax
make_pipe = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [27]:
pipe.fit(X_train,y_train)

,steps,"[('trf1', ...), ('trf2', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [28]:
y_pred = pipe.predict(X_test)

In [29]:
accuracy_score(y_test,y_pred)

0.6256983240223464

In [30]:
from sklearn.model_selection import cross_val_score

cross_val_score(pipe,X_train,y_train,cv=5, scoring='accuracy').mean()

np.float64(0.6391214419383433)

GridSearch using Pipeline

In [33]:
params = {
    'trf5__max_depth' :[1,2,3,4,5,None]
}

In [34]:
from sklearn.model_selection import  GridSearchCV

grid = GridSearchCV(pipe,param_grid=params,cv=5,scoring='accuracy')
grid.fit(X_train,y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'trf5__max_depth': [1, 2, ...]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('impute_age', ...), ('impute_embarked', ...)]"


In [35]:
grid.best_score_

np.float64(0.6391214419383433)

In [36]:
grid.best_params_

{'trf5__max_depth': 2}

In [37]:
import pickle

In [38]:
pickle.dump(pipe, open('models/pipe.pkl', 'wb'))